# VDM: train a gun/knife detector with reviewed hard negatives

**Goal:** reduce gun false positives on phones, laptops and ordinary objects while measuring how many real guns are missed. This notebook prepares data, trains a replacement, calibrates on validation, and exports a candidate. It does not modify VDM or call anyone.

**Start:** Runtime → Change runtime type → T4 GPU. Run cells in order. Stop at the review UI, inspect images, then continue. Drive holds reviews/checkpoints; Colab scratch holds images. Budget several GB and preparation time: the official train annotation CSV is ~2.26 GB. Free Colab GPU availability/runtime is not guaranteed.

Default: fine-tune **fresh COCO-pretrained YOLOv8s**, not the unreliable specialist. Set `TRAIN_FROM_SCRATCH=True` only for a separate comparison with more training/data. Class IDs stay `0: guns`, `1: knife` to match VDM. A negative has an **empty label file**, not a third background class.

Sources: [Open Images downloads](https://storage.googleapis.com/openimages/web/download_v7.html), [licenses](https://storage.googleapis.com/openimages/web/factsfigures_v7.html#licenses), [YOLO training](https://docs.ultralytics.com/modes/train/), [negative images and labels](https://docs.ultralytics.com/yolov5/tutorials/train-custom-data).

Open Images supplies guns/knives plus candidate phone/laptop/remote/TV/tablet/drill/scissors images. Missing weapon annotations do **not** prove an image is negative: only your explicit review admits negatives. Public boxes also need a visual audit. The current detector's errors on your actual camera are the most useful additions. Never mark a visible gun or knife as background; exclude ambiguous examples.

More data: [YouTube-GDD](https://github.com/UCAS-GYX/YouTube-GDD) offers 5,000 annotated gun images; use its pre-extracted image link and remap `1: gun` to our `0: guns`. Its `0: person` is not a gun. Split by original video, because its published folds can share a source video. [CCTV-Gun](https://github.com/srikarym/CCTV-Gun/blob/main/dataset_instructions.md) is useful for a later external CCTV evaluation. These additional datasets are **not** silently downloaded or mixed into test here; check their separate source terms.


In [ ]:
%pip install -q "ultralytics==8.4.148" "ipywidgets>=8,<9"


## 1. Configuration and persistent output
The package pin matches the VDM runtime used to prepare this notebook. Keep Colab's working CUDA/PyTorch installation. `s` has more capacity than the current `n`; measure its CPU latency before deployment. Use a fresh `RUN_NAME` after any data/setting changes; resume only the same run.


In [ ]:
from pathlib import Path
import base64, csv, hashlib, io, json, math, os, random, shutil, sys, time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from urllib.request import Request, urlopen
import cv2
import numpy as np
import torch
import yaml
from PIL import Image, ImageDraw
from IPython.display import display, clear_output
from tqdm.auto import tqdm
from ultralytics import YOLO, settings

USE_DRIVE = True
SEED = 42
MODEL_SIZE = "s"             # "n" for lower inference cost; "s" for this first experiment
TRAIN_FROM_SCRATCH = False
EPOCHS = 80                  # Increase for a separate scratch experiment
BATCH = 8                   # If GPU runs out of memory, reduce to 4
IMAGE_SIZE = 640             # Match VDM's inference resolution
RUN_NAME = "weapons_v1"
RESUME = False              # True only after reconnecting to the SAME dataset/run
COMPARE_CURRENT_MODEL = True
MIN_HANDGUN_RECALL = 0.75
LOCAL_HANDGUN_SESSIONS = set() # Optional: sessions where ALL gun boxes were checked to be handguns.
MIN_GUN_RECALL = 0.75        # Experimental acceptance targets, not safety guarantees
MAX_NEGATIVE_GUN_RATE = 0.02 # Fraction of weapon-free images, NOT false alerts/hour
MIN_NEGATIVES = {"train": 100, "val": 50, "test": 50}
NEGATIVE_COUNTS = {"train": 240, "val": 100, "test": 100}
POSITIVE_LIMITS = {
    "train": {"Handgun": 800, "Rifle": 400, "Shotgun": 200, "Knife": 500, "Kitchen knife": 300},
    "val": {"Handgun": 80, "Rifle": 60, "Shotgun": 40, "Knife": 80, "Kitchen knife": 60},
    "test": {"Handgun": 100, "Rifle": 80, "Shotgun": 50, "Knife": 100, "Kitchen knife": 80},
}
assert MODEL_SIZE in {"n", "s"}
if USE_DRIVE:
    from google.colab import drive, output
    drive.mount("/content/drive")
    output.enable_custom_widget_manager()
SAVE = Path("/content/drive/MyDrive/vdma-weapon-training" if USE_DRIVE else "/content/vdma-weapon-output")
WORK = Path("/content/vdma-weapon-work")
USER_DATA = SAVE / "user_data"
CACHE = SAVE / "annotation_cache"
for path in [SAVE, WORK, USER_DATA, CACHE]: path.mkdir(parents=True, exist_ok=True)
for split in ("train", "val", "test"):
    for kind in ("negative_clips", "negative_images"):
        (USER_DATA/kind/split).mkdir(parents=True, exist_ok=True)
    for kind in ("images", "labels"):
        (USER_DATA/"positives"/split/kind).mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
settings.update({"wandb": False, "comet": False, "mlflow": False, "clearml": False, "sync": False})
print("Your optional footage goes in:", USER_DATA)
print("Outputs/checkpoints:", SAVE)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — select a GPU before training")


## 2. Download helpers and official Open Images subset
The V7 site points to V6 train boxes and V5 validation/test boxes; those are intentional. We discard images with group/depiction/inside target annotations, preserve occluded/truncated physical targets, and retain all target boxes for selected images. Filtered caches persist; the multi-GB CSV is removed after filtering. Downloads fail visibly or appear in a failure manifest; failed images do not become negatives.


In [ ]:
TARGET_MIDS = {"/m/0gxl3": (0, "Handgun"), "/m/06c54": (0, "Rifle"),
    "/m/06nrc": (0, "Shotgun"), "/m/04ctx": (1, "Knife"), "/m/058qzx": (1, "Kitchen knife")}
NEGATIVE_MIDS = {"/m/050k8": "Mobile phone", "/m/01c648": "Laptop", "/m/0bh9flk": "Tablet computer",
    "/m/0qjjc": "Remote control", "/m/07c52": "Television", "/m/01d380": "Drill (Tool)", "/m/01lsmm": "Scissors"}
BOX_URLS = {
    "train": "https://storage.googleapis.com/openimages/v6/oidv6-train-annotations-bbox.csv",
    "val": "https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv",
    "test": "https://storage.googleapis.com/openimages/v5/test-annotations-bbox.csv"}
META_URLS = {
    "train": "https://storage.googleapis.com/openimages/2018_04/train/train-images-boxable-with-rotation.csv",
    "val": "https://storage.googleapis.com/openimages/2018_04/validation/validation-images-with-rotation.csv",
    "test": "https://storage.googleapis.com/openimages/2018_04/test/test-images-with-rotation.csv"}
NAMES = ["guns", "knife"]

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".part")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2))
    temp.replace(path)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024*1024), b""): digest.update(block)
    return digest.hexdigest()

def download(url, path, headers=None):
    path = Path(path)
    if path.is_file() and path.stat().st_size: return path
    path.parent.mkdir(parents=True, exist_ok=True)
    part = path.with_suffix(path.suffix + ".part")
    for attempt in range(3):
        try:
            with urlopen(Request(url, headers=headers or {}), timeout=60) as response, part.open("wb") as out:
                expected = int(response.headers.get("Content-Length", 0))
                count = 0
                while block := response.read(1024*1024):
                    out.write(block)
                    count += len(block)
                if expected and count != expected: raise IOError("Incomplete download")
            part.replace(path)
            return path
        except Exception:
            part.unlink(missing_ok=True)
            if attempt == 2: raise
            time.sleep(attempt + 1)

def valid_box(box):
    cls, x1, y1, x2, y2 = box
    return cls in (0, 1) and all(math.isfinite(v) for v in box) and 0 <= x1 < x2 <= 1 and 0 <= y1 < y2 <= 1

def read_oi_subset(path):
    targets, topics, negative = defaultdict(list), defaultdict(set), defaultdict(set)
    handguns = defaultdict(list)
    blocked, weapon_ids = set(), set()
    with Path(path).open(newline="") as handle:
        for row in csv.DictReader(handle):
            mid, image_id = row["LabelName"], row["ImageID"]
            if mid in TARGET_MIDS:
                weapon_ids.add(image_id)  # Even a rejected annotation disqualifies an automatic negative candidate.
                cls, topic = TARGET_MIDS[mid]
                box = [cls] + [float(row[key]) for key in ("XMin", "YMin", "XMax", "YMax")]
                if any(int(row.get(key) or 0) for key in ("IsGroupOf", "IsDepiction", "IsInside")) or not valid_box(box):
                    blocked.add(image_id)
                    continue
                targets[image_id].append(box)
                if mid == "/m/0gxl3": handguns[image_id].append(box)
                topics[image_id].add(topic)
            elif mid in NEGATIVE_MIDS:
                negative[NEGATIVE_MIDS[mid]].add(image_id)
    return {"targets": {key: [list(b) for b in sorted(set(tuple(b) for b in boxes))]
                        for key, boxes in targets.items() if key not in blocked},
            "topics": {key: sorted(value) for key, value in topics.items() if key not in blocked},
            "handguns": {key: value for key, value in handguns.items() if key not in blocked},
            "negative": {key: sorted(value - weapon_ids) for key, value in negative.items()}}

def oi_subset(split):
    cache = CACHE / f"oi-boxes-v2-{split}.json"
    if cache.exists(): return json.loads(cache.read_text())
    print("Filtering official annotations:", split, "(the train file is about 2.26 GB)")
    path = download(BOX_URLS[split], WORK/f"{split}-boxes.csv")
    subset = read_oi_subset(path)
    subset["annotation_sha256"] = sha256_file(path)
    subset["annotation_url"] = BOX_URLS[split]
    save_json(cache, subset)
    path.unlink()  # Keep compact provenance/filter cache, not the multi-GB CSV.
    return subset

def select_oi(split, subset, limits, negatives):
    rank = lambda key: hashlib.sha256(f"{SEED}:{split}:{key}".encode()).hexdigest()
    chosen = set()
    for topic, limit in limits.items():
        available = [key for key in subset["targets"] if topic in subset["topics"][key]]
        chosen.update(sorted(available, key=rank)[:limit])
        print(split, topic, "available", len(available), "requested", limit)
    selected_negative = {}
    pools = {topic: iter(sorted(ids, key=rank)) for topic, ids in subset["negative"].items()}
    while pools and len(selected_negative) < negatives:
        for topic in list(pools):
            image_id = next(pools[topic], None)
            if image_id is None: del pools[topic]
            elif image_id not in chosen: selected_negative.setdefault(image_id, topic)
            if len(selected_negative) >= negatives: break
    records = []
    remote_split = "validation" if split == "val" else split
    for image_id in sorted(chosen | set(selected_negative)):
        boxes = subset["targets"].get(image_id, [])
        records.append({"id": f"oi-{remote_split}-{image_id}", "image_id": image_id,
            "source": "open_images", "split": split, "group": "oi:"+image_id,
            "boxes": boxes, "handgun_boxes": subset["handguns"].get(image_id, []), "topic": ", ".join(subset["topics"][image_id]) if boxes else selected_negative[image_id],
            "url": f"https://open-images-dataset.s3.amazonaws.com/{remote_split}/{image_id}.jpg",
            "path": str(WORK/"images"/f"oi-{remote_split}-{image_id}.jpg")})
    return records

def download_image(record):
    path = download(record["url"], record["path"])
    try:
        with Image.open(path) as im: im.verify()
    except Exception:
        path.unlink(missing_ok=True)
        raise
    return record


In [ ]:
# Verify label IDs against Google's current published boxable class list.
class_file = download("https://storage.googleapis.com/openimages/v7/oidv7-class-descriptions-boxable.csv", WORK/"classes.csv")
with class_file.open(newline="") as handle: published_names = dict(csv.reader(handle))
for mid, (_, name) in TARGET_MIDS.items(): assert published_names[mid] == name
for mid, name in NEGATIVE_MIDS.items(): assert published_names[mid] == name

public_records, annotation_sources = [], {}
for split in ("train", "val", "test"):
    subset = oi_subset(split)
    annotation_sources[split] = {key: subset[key] for key in ("annotation_url", "annotation_sha256")}
    public_records += select_oi(split, subset, POSITIVE_LIMITS[split], NEGATIVE_COUNTS[split])
failures, downloaded = [], []
def fetch_one(record):
    try: return download_image(record), None
    except Exception as exc: return None, {"id": record["id"], "error": type(exc).__name__}
with ThreadPoolExecutor(max_workers=8) as pool:
    for record, error in tqdm(pool.map(fetch_one, public_records), total=len(public_records)):
        if error: failures.append(error)
        else: downloaded.append(record)
public_records = downloaded
save_json(SAVE/"download_failures.json", failures)
print("Downloaded:", len(public_records), "Failures:", len(failures))
print("Public source counts:", Counter((r["split"], bool(r["boxes"])) for r in public_records))


## 3. Optional: your actual camera negatives and additional labelled positives
Copy your files into `USER_DATA` in Drive using this layout. Keep **all clips from one recording session in the same split and session folder**. Use different rooms/days/people for validation/test. Merely giving two clips different filenames does not make them independent.

```text
negative_clips/train/session_a/phone_laptop.mp4
negative_clips/val/session_b/different_day.mp4
negative_clips/test/session_c/different_room.mp4
negative_images/train/session_d/photo.jpg
positives/train/images/session_e/frame.jpg
positives/train/labels/session_e/frame.txt
```

Positive files must use `0: guns`, `1: knife`, with **every** visible target boxed. Empty/missing positive labels fail; put weapon-free examples in negative folders. For YouTube-GDD, reassign by original video ID, remove person boxes, remap gun, and check for unlabelled knives before importing. Negative videos are sampled every two seconds (max 40 frames/clip), then reviewed below. These limits prevent thousands of nearly identical frames dominating training. Local footage is not uploaded anywhere by this notebook beyond your own selected Drive/Colab storage.


In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp"}
VIDEO_SUFFIXES = {".mp4", ".mov", ".avi", ".mkv"}

def read_yolo_labels(path):
    if not path.is_file(): raise ValueError(f"Missing positive label: {path}")
    boxes = []
    for line in path.read_text().splitlines():
        values = [float(x) for x in line.split()]
        if len(values) != 5 or values[0] not in (0, 1): raise ValueError(f"Expected guns/knife YOLO box: {path}")
        cls, cx, cy, w, h = values
        box = [int(cls), cx-w/2, cy-h/2, cx+w/2, cy+h/2]
        # Permit rounding at image borders, not invalid/negative boxes.
        if min(box[1:]) >= -1e-6 and max(box[1:]) <= 1+1e-6:
            box[1:] = [min(1., max(0., x)) for x in box[1:]]
        if not valid_box(box): raise ValueError(f"Invalid box: {path}")
        boxes.append(box)
    if not boxes: raise ValueError(f"Positive label is empty; review it as a negative instead: {path}")
    return boxes

def local_record(path, split, session, boxes, topic, identity):
    return {"id": "local-"+identity, "image_id": None, "source": "local",
            "split": split, "group": "session:"+session, "boxes": boxes,
            "handgun_boxes": [b for b in boxes if b[0] == 0] if session in LOCAL_HANDGUN_SESSIONS else [],
            "topic": topic, "url": None, "path": str(path)}

def import_local(root):
    records, sessions, media_splits = [], {}, {}
    for split in ("train", "val", "test"):
        for kind in ("negative_images", "negative_clips", "positives"):
            base = root/kind/split/("images" if kind == "positives" else "")
            for path in sorted(base.rglob("*")):
                allowed = VIDEO_SUFFIXES if kind == "negative_clips" else IMAGE_SUFFIXES
                if not path.is_file() or path.suffix.lower() not in allowed: continue
                relative = path.relative_to(base)
                if len(relative.parts) < 2: raise ValueError(f"Put files inside a session folder: {path}")
                session = relative.parts[0]
                if sessions.setdefault(session, split) != split: raise ValueError(f"Session crosses splits: {session}")
                digest = sha256_file(path)
                if media_splits.setdefault(digest, split) != split: raise ValueError("Identical media crosses splits")
                if kind == "positives":
                    labels = root/kind/split/"labels"/relative.with_suffix(".txt")
                    records.append(local_record(path, split, session, read_yolo_labels(labels), "local positive", digest))
                elif kind == "negative_images":
                    records.append(local_record(path, split, session, [], "local hard negative", digest))
                else:
                    cap = cv2.VideoCapture(str(path))
                    try:
                        fps, count = cap.get(cv2.CAP_PROP_FPS), cap.get(cv2.CAP_PROP_FRAME_COUNT)
                        if not cap.isOpened() or not math.isfinite(fps) or fps <= 0 or count <= 0:
                            raise ValueError(f"Unreadable video: {path.name}")
                        indices = np.arange(0, int(count), max(1, round(fps*2)))
                        if len(indices) > 40: indices = indices[np.linspace(0, len(indices)-1, 40).astype(int)]
                        for frame_id in indices:
                            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_id))
                            ok, frame = cap.read()
                            if not ok: raise ValueError(f"Failed frame {frame_id}: {path.name}")
                            target = WORK/"local_frames"/f"{digest}-{frame_id}.jpg"
                            target.parent.mkdir(parents=True, exist_ok=True)
                            if not cv2.imwrite(str(target), frame): raise IOError("Frame write failed")
                            records.append(local_record(target, split, session, [], "local video hard negative", f"{digest}-{frame_id}"))
                    finally: cap.release()
    return records

local_records = import_local(USER_DATA)
records = public_records + local_records
print("Local examples:", Counter((r["split"], bool(r["boxes"])) for r in local_records))
if not local_records: print("No local footage supplied: this run cannot establish that your phone/laptop problem is fixed.")


## 4. Split audit and duplicate removal
Exact decoded-pixel duplicates are removed. A conservative 64-bit difference-hash check removes near duplicates **across** splits, keeping test ahead of validation ahead of train. It can remove some merely similar images and cannot catch all video/scene leakage. Session separation and manual audits are still required. The pairwise check is intended for this bounded few-thousand-image experiment.


In [ ]:
def image_fingerprints(path):
    with Image.open(path) as original:
        im = original.convert("RGB")
        exact = hashlib.sha256(str(im.size).encode()+im.tobytes()).hexdigest()
        small = np.asarray(im.convert("L").resize((9, 8), Image.Resampling.BILINEAR))
        bits = (small[:, 1:] > small[:, :-1]).reshape(-1)
        dhash = sum(int(bit) << i for i, bit in enumerate(bits))
    return exact, dhash

def deduplicate(records):
    groups = defaultdict(set)
    for r in records: groups[r["group"]].add(r["split"])
    if any(len(v)>1 for v in groups.values()): raise ValueError("Source group appears in multiple splits")
    seen, kept, dropped = {}, [], []
    rank = {"test": 0, "val": 1, "train": 2}
    for record in tqdm(sorted(records, key=lambda r: (rank[r["split"]], r["id"])), desc="Deduplicate"):
        r = dict(record)
        pixel_hash, perceptual = image_fingerprints(r["path"])
        r["pixel_sha256"] = pixel_hash
        # Bind reviews to the actual pixels; a changed download cannot reuse an old approval.
        r["review_key"] = r["id"]+":"+pixel_hash
        if pixel_hash in seen:
            previous = seen[pixel_hash]
            if {b[0] for b in previous["boxes"]} != {b[0] for b in r["boxes"]}:
                raise ValueError(f"Duplicate has conflicting target classes: {r['id']}; correct/exclude it first")
            dropped.append({"id": r["id"], "reason": "exact duplicate", "kept": previous["id"]})
            continue
        seen[pixel_hash] = r
        # ponytail: bounded O(n²) cross-split scan; use an indexed perceptual-hash search at larger scale.
        similar = next((other for other, value in kept if other["split"] != r["split"] and (value ^ perceptual).bit_count() <= 4), None)
        if similar:
            dropped.append({"id": r["id"], "reason": "near duplicate across splits", "kept": similar["id"]})
        else: kept.append((r, perceptual))
    return [r for r, _ in kept], dropped

records, dropped_duplicates = deduplicate(records)
save_json(SAVE/"duplicates_removed.json", dropped_duplicates)
print("Kept:", len(records), "Removed:", len(dropped_duplicates))


## 5. Review candidate negatives — required
Check each image at full size when uncertain. **Keep** means no physical gun **and no knife anywhere** in the image. Unticked images are excluded; unchecked/unfinished pages are never assumed safe. If a weapon appears, exclude it here and add it to positives only after drawing correct boxes. Choices save immediately to Drive and survive reconnects.

The split names are hidden in this UI: establish labels without looking at model predictions or tuning to test results. Public data includes hunting/weapon scenes; do not accept a page without looking. For small/distant objects, click **View full image** and exclude uncertain cases.


In [ ]:
import ipywidgets as widgets
REVIEW_FILE = SAVE/"negative_reviews.json"
reviews = json.loads(REVIEW_FILE.read_text()) if REVIEW_FILE.exists() else {}
negative_candidates = [r for r in records if not r["boxes"]]
review_output = widgets.Output()

def show_negative_page():
    pending = [r for r in negative_candidates if reviews.get(r["review_key"]) not in ("keep", "exclude")]
    with review_output:
        clear_output(wait=True)
        print(f"Reviewed {len(negative_candidates)-len(pending)} / {len(negative_candidates)} candidates")
        if not pending:
            print("Review complete. Continue to the positive audit and dataset build.")
            return
        page, choices, cards = pending[:9], [], []
        for r in page:
            with Image.open(r["path"]) as im:
                thumbnail = im.convert("RGB")
                thumbnail.thumbnail((300, 220))
                buffer = io.BytesIO(); thumbnail.save(buffer, format="JPEG")
            check = widgets.Checkbox(value=False, description="No gun AND no knife", indent=False)
            button = widgets.Button(description="View full image")
            full = widgets.Output()
            def expand(_button, path=r["path"], out=full):
                with out:
                    clear_output(wait=True)
                    display(Image.open(path))
            button.on_click(expand)
            choices.append(check)
            cards.append(widgets.VBox([widgets.Image(value=buffer.getvalue()), widgets.Label(r["topic"]), check, button, full]))
        submit = widgets.Button(description="Save page: keep checked, exclude others", layout=widgets.Layout(width="360px"), button_style="primary")
        def accept(_):
            for r, check in zip(page, choices): reviews[r["review_key"]] = "keep" if check.value else "exclude"
            save_json(REVIEW_FILE, reviews)
            show_negative_page()
        submit.on_click(accept)
        display(widgets.GridBox(cards, layout=widgets.Layout(grid_template_columns="repeat(3, minmax(0, 1fr))")), submit)

display(review_output)
show_negative_page()


## 6. Audit positive boxes
This displays a deterministic sample from every source/split/class combination, plus imported local positives. Check the object is real, all visible targets are boxed, and boxes follow object edges. Add bad IDs to `EXCLUDE_IDS` below; correct annotations externally if needed. This sample is a minimum audit, not a guarantee all public labels are correct. For a production dataset review all annotations, especially small guns, occlusion and missed second weapons.


In [ ]:
def annotated_image(record):
    with Image.open(record["path"]) as original: im = original.convert("RGB")
    draw = ImageDraw.Draw(im)
    for cls, x1, y1, x2, y2 in record["boxes"]:
        box = (x1*im.width, y1*im.height, x2*im.width, y2*im.height)
        draw.rectangle(box, outline="red" if cls == 0 else "cyan", width=max(2, im.width//300))
        draw.text((box[0], max(0, box[1]-12)), NAMES[cls], fill="red" if cls == 0 else "cyan")
    im.thumbnail((700, 450))
    return im

positive_audit = {}
for source in ("open_images", "local"):
    for split in ("train", "val", "test"):
        for cls in (0, 1):
            candidates = [r for r in records if r["source"] == source and r["split"] == split and any(b[0] == cls for b in r["boxes"])]
            for r in random.Random(SEED).sample(candidates, min(6, len(candidates))): positive_audit[r["id"]] = r
for r in positive_audit.values():
    print(r["id"], r["topic"])
    display(annotated_image(r))


In [ ]:
PUBLIC_BOX_AUDIT_OK = False # Change to True only after checking the displayed annotations.
LOCAL_POSITIVE_LABELS_REVIEWED = False # True if you imported local positives and checked all their boxes.
EXCLUDE_IDS = set() # Example: {"oi-train-badimageid"}; excluded from all subsequent steps.
assert PUBLIC_BOX_AUDIT_OK, "Inspect the positive audit, correct/exclude errors, then set PUBLIC_BOX_AUDIT_OK=True."
assert LOCAL_POSITIVE_LABELS_REVIEWED or not any(r["source"] == "local" and r["boxes"] for r in records), "Review every imported positive label first."
reviews = json.loads(REVIEW_FILE.read_text()) if REVIEW_FILE.exists() else {}
curated = [r for r in records if r["id"] not in EXCLUDE_IDS and (r["boxes"] or reviews.get(r["review_key"]) == "keep")]
for split in ("train", "val", "test"):
    subset = [r for r in curated if r["split"] == split]
    negatives = sum(not r["boxes"] for r in subset)
    assert negatives >= MIN_NEGATIVES[split], f"{split}: need at least {MIN_NEGATIVES[split]} reviewed negatives; have {negatives}. Add/review more."
    for cls in (0, 1):
        count = sum(any(b[0] == cls for b in r["boxes"]) for r in subset)
        assert count >= (30 if split == "train" else 10), f"Too few {NAMES[cls]} positive images in {split}: {count}"
    print(split, "images", len(subset), "reviewed negatives", negatives)
print("Unreviewed negatives excluded:", sum(not r["boxes"] and reviews.get(r["review_key"]) not in ("keep", "exclude") for r in records))


## 7. Export clean YOLO data, provenance and attribution
Dataset folders are addressed by their manifest hash. Rerunning with different reviews creates a different dataset, so stale files cannot silently remain in training. Attribution metadata for selected Open Images is retained with the exported manifest; verify image rights before redistributing data. The notebook never packages the images into the model download.


In [ ]:
def label_text(boxes):
    lines = []
    for box in boxes:
        if not valid_box(box): raise ValueError("Invalid training box")
        cls, x1, y1, x2, y2 = box
        lines.append(f"{cls} {(x1+x2)/2:.8f} {(y1+y2)/2:.8f} {x2-x1:.8f} {y2-y1:.8f}")
    return "\n".join(lines) + ("\n" if lines else "")

manifest = [{k: v for k, v in r.items() if k not in ("path", "review_key")} for r in sorted(curated, key=lambda r:r["id"])]
dataset_hash = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
DATA = WORK/"datasets"/dataset_hash[:16]
for split in ("train", "val", "test"):
    for kind in ("images", "labels"): (DATA/kind/split).mkdir(parents=True, exist_ok=True)
for r in tqdm(curated, desc="Write YOLO dataset"):
    dst = DATA/"images"/r["split"]/(r["id"]+".jpg")
    with Image.open(r["path"]) as im: im.convert("RGB").save(dst, quality=95)
    (DATA/"labels"/r["split"]/(r["id"]+".txt")).write_text(label_text(r["boxes"]))
    r["training_path"] = str(dst)
DATA_YAML = DATA/"data.yaml"
DATA_YAML.write_text(yaml.safe_dump({"path": str(DATA), "train": "images/train", "val": "images/val", "test": "images/test", "names": dict(enumerate(NAMES))}, sort_keys=False))
save_json(SAVE/f"dataset-{dataset_hash[:16]}.json", {"sha256": dataset_hash, "annotations": annotation_sources, "images": manifest})

attribution = []
for split in ("train", "val", "test"):
    ids = {r["image_id"] for r in curated if r["source"] == "open_images" and r["split"] == split}
    cache = CACHE/f"attribution-{split}-{hashlib.sha256(''.join(sorted(ids)).encode()).hexdigest()[:16]}.json"
    if cache.exists(): rows = json.loads(cache.read_text())
    else:
        rows = []
        path = download(META_URLS[split], WORK/f"{split}-metadata.csv")
        with path.open(newline="") as handle:
            for row in csv.DictReader(handle):
                if row["ImageID"] in ids: rows.append(row)
        save_json(cache, rows)
        path.unlink()
    assert {r["ImageID"] for r in rows} == ids, "Missing selected-image attribution; inspect the official metadata."
    attribution += rows
save_json(SAVE/f"attribution-{dataset_hash[:16]}.json", attribution)
print("Dataset:", DATA_YAML, "fingerprint:", dataset_hash)


## 8. Train (or resume the same experiment)
Training starts only after the data checks above. Checkpoints save to Drive every five epochs and at the latest completed epoch. Free Colab can disconnect; reconnect, rerun preparation with the same settings/reviews, set `RESUME=True`, and run this cell. We check the dataset fingerprint before resuming. Do not change train/validation/test membership between a run and its resume.

`best.pt` is selected by YOLO validation fitness, not by your false-alarm target; the next cells measure that target explicitly. Training from random weights is an optional experiment, not a prerequisite to training your own detector.


In [ ]:
assert torch.cuda.is_available(), "Choose a GPU runtime; full training is not started on CPU."
RUN = SAVE/"runs"/RUN_NAME
run_config = {"dataset_sha256": dataset_hash, "seed": SEED, "model_size": MODEL_SIZE,
    "from_scratch": TRAIN_FROM_SCRATCH, "image_size": IMAGE_SIZE, "epochs": EPOCHS, "batch": BATCH,
    "names": NAMES, "ultralytics": __import__("ultralytics").__version__}
if RESUME:
    assert (RUN/"weights/last.pt").exists(), "No checkpoint to resume."
    assert json.loads((RUN/"experiment.json").read_text()) == run_config, "Dataset/settings changed; use a new RUN_NAME."
    model = YOLO(str(RUN/"weights/last.pt"))
    model.train(resume=True, device=0)
else:
    assert not RUN.exists(), "Run already exists: set RESUME=True or choose a new RUN_NAME."
    RUN.mkdir(parents=True)
    save_json(RUN/"experiment.json", run_config)
    initializer = f"yolov8{MODEL_SIZE}.yaml" if TRAIN_FROM_SCRATCH else f"yolov8{MODEL_SIZE}.pt"
    model = YOLO(initializer)
    if not TRAIN_FROM_SCRATCH:
        initial_path = Path(model.ckpt_path)
        save_json(RUN/"initial_weights.json", {"source": "official Ultralytics COCO checkpoint", "file": initial_path.name, "sha256": sha256_file(initial_path)})
    model.train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH, device=0,
        project=str(RUN.parent), name=RUN.name, exist_ok=True, seed=SEED, deterministic=True,
        pretrained=not TRAIN_FROM_SCRATCH, workers=2, cache=False, patience=20, save_period=5,
        optimizer="AdamW", lr0=0.001, mosaic=0.5, close_mosaic=10, mixup=0.0,
        degrees=8.0, translate=0.1, scale=0.3, fliplr=0.5, flipud=0.0, plots=True)
BEST = RUN/"weights/best.pt"
assert BEST.exists()
candidate = YOLO(str(BEST))
assert [candidate.names[i] for i in range(len(candidate.names))] == NAMES
save_json(RUN/"environment.json", {"python": sys.version, "torch": torch.__version__,
    "ultralytics": __import__("ultralytics").__version__, "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0)})
import subprocess
(RUN/"pip-freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
print("Candidate:", BEST)


## 9. Validation: measure false alarms AND recall
We evaluate at the app's 640px input size and `max_det=50`. Threshold selection sees **validation only**. Box precision/recall use same-class one-to-one matching at IoU ≥ 0.5; negative false-positive rate counts reviewed weapon-free images with at least one gun prediction. It is not false alerts/hour, and correlated frames reduce the effective sample size.

The example targets require ≤2% negative-image gun alarms, ≥75% gun box recall, and ≥75% handgun box recall with at least ten handgun validation images. Open Images has only 17 usable handgun validation images before duplicate filtering in the checked release: this is a small starting benchmark, not enough to establish deployment quality. Add independently labelled handgun examples for a stronger assessment. `LOCAL_HANDGUN_SESSIONS` can identify imported sessions whose gun boxes you verified are all handguns; mixed/unknown gun types are left out of that subtype metric. If no threshold meets both, the notebook returns **no recommendation**. Do not fix false positives by silently setting a threshold that misses all guns. Larger, varied negatives are needed to estimate rare false alarms reliably. Knife behavior is also evaluated at the app's current 0.50 threshold.


In [ ]:
def predict_records(model, rows):
    predictions = {}
    # Bound host memory: Ultralytics may load a list of image paths eagerly.
    for start in tqdm(range(0, len(rows), 16), desc="Inference"):
        batch_rows = rows[start:start+16]
        results = model.predict([r["training_path"] for r in batch_rows], stream=True, batch=8,
            imgsz=IMAGE_SIZE, conf=0.01, iou=0.7, max_det=50, device=0, verbose=False)
        for row, result in zip(batch_rows, results):
            predictions[row["id"]] = [[int(c), float(p), *[float(x) for x in box]]
                for c, p, box in zip(result.boxes.cls.cpu().tolist(), result.boxes.conf.cpu().tolist(), result.boxes.xyxyn.cpu().tolist())]
    assert len(predictions) == len(rows), "Incomplete prediction results"
    return predictions

def box_iou(a, b):
    intersection = max(0., min(a[2], b[2])-max(a[0], b[0])) * max(0., min(a[3], b[3])-max(a[1], b[1]))
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - intersection
    return intersection/union if union > 0 else 0.

def wilson_upper(k, n):
    if not n: return None
    z, p = 1.96, k/n
    return (p+z*z/(2*n)+z*math.sqrt(p*(1-p)/n+z*z/(4*n*n)))/(1+z*z/n)

def score_at(rows, predictions, threshold, cls=0):
    tp = fp = fn = negatives = negative_fp = 0
    by_topic = defaultdict(lambda: {"images": 0, "false_positive_images": 0})
    for row in rows:
        truth = [b[1:] for b in row["boxes"] if b[0] == cls]
        det = sorted((p for p in predictions[row["id"]] if p[0] == cls and p[1] >= threshold), key=lambda p: -p[1])
        matched = set()
        for prediction in det:
            options = [(box_iou(prediction[2:], box), i) for i, box in enumerate(truth) if i not in matched]
            overlap, index = max(options, default=(0., -1))
            if overlap >= 0.5:
                tp += 1
                matched.add(index)
            else: fp += 1
        fn += len(truth)-len(matched)
        if not row["boxes"]:
            negatives += 1
            negative_fp += bool(det)
            by_topic[row["topic"]]["images"] += 1
            by_topic[row["topic"]]["false_positive_images"] += bool(det)
    precision = tp/(tp+fp) if tp+fp else 0.
    recall = tp/(tp+fn) if tp+fn else 0.
    return {"class": NAMES[cls], "threshold": float(threshold), "tp": tp, "fp": fp, "fn": fn,
        "precision_iou50": precision, "recall_iou50": recall,
        "f1_iou50": 2*precision*recall/(precision+recall) if precision+recall else 0.,
        "negative_images": negatives, "negative_false_positive_images": negative_fp,
        "negative_image_fp_rate": negative_fp/negatives if negatives else None,
        "negative_image_fp_wilson_upper95": wilson_upper(negative_fp, negatives), "negative_topics": dict(by_topic)}

def score_guns(rows, predictions, threshold):
    score = score_at(rows, predictions, threshold)
    handgun_rows = [{**r, "boxes": r["handgun_boxes"]} for r in rows if r.get("handgun_boxes")]
    handgun = score_at(handgun_rows, predictions, threshold)
    score.update(handgun_images=len(handgun_rows), handgun_boxes=handgun["tp"]+handgun["fn"],
                 handgun_recall_iou50=handgun["recall_iou50"] if handgun_rows else None)
    return score

def choose_threshold(scores, max_fp, min_recall, min_handgun_recall=None):
    eligible = [s for s in scores if s["negative_image_fp_rate"] is not None
                and s["negative_image_fp_rate"] <= max_fp and s["recall_iou50"] >= min_recall
                and (min_handgun_recall is None or s.get("handgun_images", 0) >= 10
                     and s["handgun_recall_iou50"] >= min_handgun_recall)]
    return max(eligible, key=lambda s: (s["recall_iou50"], s["precision_iou50"]), default=None)

validation_rows = [r for r in curated if r["split"] == "val"]
validation_predictions = predict_records(candidate, validation_rows)
threshold_scores = [score_guns(validation_rows, validation_predictions, float(t)) for t in np.arange(.10, .951, .025)]
selection = choose_threshold(threshold_scores, MAX_NEGATIVE_GUN_RATE, MIN_GUN_RECALL, MIN_HANDGUN_RECALL)
GUN_THRESHOLD = selection["threshold"] if selection else 0.50
print("Recommended gun threshold:", selection["threshold"] if selection else "NONE: neither an improved detector nor a deployable cutoff is established")
print("If none, 0.50 is used below only to report the failure, not as a recommendation.")
print(json.dumps(score_guns(validation_rows, validation_predictions, GUN_THRESHOLD), indent=2))
print("Knife at 0.50:", json.dumps(score_at(validation_rows, validation_predictions, .50, cls=1), indent=2))
save_json(RUN/"validation_thresholds.json", {"selection": selection, "scores": threshold_scores})
standard_val = candidate.val(data=str(DATA_YAML), split="val", imgsz=IMAGE_SIZE, batch=BATCH, device=0,
    conf=0.001, max_det=50, plots=True, project=str(RUN), name="validation_metrics")
save_json(RUN/"validation_map.json", {key: float(value) for key, value in standard_val.results_dict.items()})


## 10. Final test and comparison to today's model
Run this after freezing the checkpoint and cutoff. Do not use these test errors to tune/retrain and still call the same split an untouched test set. Collect a new independent test set for the next iteration. Test results describe this sampled dataset; they do not prove reliability in your college camera scenes.

The optional baseline downloads VDM's exact pinned Assalim checkpoint and verifies its SHA-256. It is loaded with the app's restricted `weights_only=True` allowlist, without executing repository code. Its [repository license](https://github.com/JoaoAssalim/Weapons-and-Knives-Detector-with-YOLOv8/blob/3d641e6001abdaa1afa3cd45d0fe02a9554f3d0e/LICENSE) is GPL-3.0. The baseline is for comparison, not initialization.


In [ ]:
test_rows = [r for r in curated if r["split"] == "test"]
test_predictions = predict_records(candidate, test_rows)
report = {"dataset_sha256": dataset_hash, "candidate_sha256": sha256_file(BEST),
    "validation_targets_met": selection is not None, "recommended_gun_threshold": selection["threshold"] if selection else None,
    "candidate_at_selected_or_fallback_threshold": score_guns(test_rows, test_predictions, GUN_THRESHOLD),
    "candidate_at_original_050": score_guns(test_rows, test_predictions, .50),
    "candidate_knife_at_050": score_at(test_rows, test_predictions, .50, cls=1),
    "has_local_test_negatives": any(r["source"] == "local" and not r["boxes"] for r in test_rows)}
standard_test = candidate.val(data=str(DATA_YAML), split="test", imgsz=IMAGE_SIZE, batch=BATCH, device=0,
    conf=0.001, max_det=50, plots=True, project=str(RUN), name="test_metrics")
report["candidate_test_map"] = {key: float(value) for key, value in standard_test.results_dict.items()}

if COMPARE_CURRENT_MODEL:
    baseline_path = download("https://api.github.com/repos/JoaoAssalim/Weapons-and-Knives-Detector-with-YOLOv8/git/blobs/27bc7a0b92d72282ec7c12472395e3c6d0c0edbd",
        WORK/"assalim-current.pt", {"Accept": "application/vnd.github.raw+json", "User-Agent": "VDM-training-comparison"})
    assert sha256_file(baseline_path) == "21d61ad8068caca3062d33fd8d05da445ef9ae81a2bb817249e085b0f1307cf0", "Baseline hash mismatch; do not load it."
    from ultralytics.nn.tasks import DetectionModel
    from ultralytics.nn.modules import Bottleneck, SPPF, DFL, Concat, Detect, Conv, C2f
    from ultralytics.utils import IterableSimpleNamespace
    from ultralytics.utils.loss import v8DetectionLoss, BboxLoss
    from ultralytics.utils.tal import TaskAlignedAssigner
    allowed = [DetectionModel, Bottleneck, SPPF, DFL, Concat, Detect, Conv, C2f,
        torch.nn.Conv2d, torch.nn.BatchNorm2d, torch.nn.SiLU, torch.nn.MaxPool2d,
        torch.nn.Upsample, torch.nn.ModuleList, torch.nn.Sequential, IterableSimpleNamespace,
        v8DetectionLoss, BboxLoss, TaskAlignedAssigner, torch.nn.BCEWithLogitsLoss]
    with torch.serialization.safe_globals(allowed): checkpoint = torch.load(baseline_path, map_location="cpu", weights_only=True)
    baseline = YOLO("yolov8n.yaml", task="detect", verbose=False)
    baseline.model = checkpoint["model"].float().eval()
    baseline.model.args = {}
    assert [baseline.names[i] for i in range(len(baseline.names))] == NAMES
    baseline_predictions = predict_records(baseline, test_rows)
    report["current_model_gun_at_050"] = score_guns(test_rows, baseline_predictions, .50)
    report["current_model_knife_at_050"] = score_at(test_rows, baseline_predictions, .50, cls=1)
    report["baseline_sha256"] = sha256_file(baseline_path)
    del baseline, checkpoint
print(json.dumps(report, indent=2))
save_json(RUN/"test_report.json", report)


## 11. Export the candidate for VDM
Download the ZIP and keep `best.pt`, `model_manifest.json`, `test_report.json` together. The export includes provenance/configuration but no training images or baseline weights. A model is **not deployed** just because training completed. Check both gun false alarms/recall and knife regressions, including longer held-out camera recordings.

VDM currently pins the old filename/hash and filters twice at 0.50. Integration must deliberately update the weight pin and both threshold gates, preserve the restricted loader and class mapping, and benchmark CPU inference. Do not overwrite the old checkpoint under its old filename or disable checksum validation. This notebook does not change the running app.

For the next experiment, collect more **training-only** phone/laptop/hand hard negatives, review labels, use a new run name, and compare again. Retain a genuinely independent test set. Training from scratch is available by switching the config; use the same splits for a fair comparison.


In [ ]:
EXPORT = RUN/"export"
EXPORT.mkdir(exist_ok=True)
shutil.copy2(BEST, EXPORT/"best.pt")
for name in ("experiment.json", "environment.json", "pip-freeze.txt", "test_report.json", "validation_thresholds.json", "validation_map.json"):
    shutil.copy2(RUN/name, EXPORT/name)
shutil.copy2(SAVE/f"dataset-{dataset_hash[:16]}.json", EXPORT/"dataset_manifest.json")
shutil.copy2(SAVE/f"attribution-{dataset_hash[:16]}.json", EXPORT/"open_images_attribution.json")
save_json(EXPORT/"model_manifest.json", {"filename": "best.pt", "sha256": sha256_file(EXPORT/"best.pt"),
    "architecture": f"YOLOv8{MODEL_SIZE}", "names": dict(enumerate(NAMES)), "image_size": IMAGE_SIZE,
    "gun_threshold": report["recommended_gun_threshold"], "knife_threshold_for_evaluation": 0.50,
    "dataset_sha256": dataset_hash, "validation_targets_met": report["validation_targets_met"],
    "status": "candidate_requires_local_camera_evaluation_and_explicit_integration",
    "sources": ["https://storage.googleapis.com/openimages/web/download_v7.html",
                "https://storage.googleapis.com/openimages/web/factsfigures_v7.html#licenses",
                "https://github.com/ultralytics/ultralytics/blob/main/LICENSE"]})
(EXPORT/"README.txt").write_text("Candidate only. Check test_report.json before integration. Preserve VDM weight checks and restricted loading. Update both threshold gates; benchmark CPU latency. No image data or baseline weights included. Open Images annotation CC BY 4.0 / images listed CC BY 2.0 with per-image verification. Ultralytics AGPL-3.0 or applicable commercial license; review upstream terms before distribution.\n")
archive = shutil.make_archive(str(RUN/"vdma-weapon-model"), "zip", EXPORT)
print("Saved:", archive)
from google.colab import files
files.download(archive)
